# HW2 on H100 (Colab)

Runs all sections end-to-end. Each cell writes its outputs to `logs/<category>/<ts>_<name>/`.

**Before you start:** Runtime → Change runtime type → **H100 GPU** → Save.

## 1. Setup

Clone, install everything, then **restart the runtime** (vLLM downgrades torch — Python kernel must re-import). Colab will prompt automatically; if not, do it manually after this cell.

In [ ]:
!git clone https://github.com/trevorbchen/148bhw2.git || (cd 148bhw2 && git pull)
%cd /content/148bhw2

# vLLM pins torch==2.5.1 — install it first so it dictates the torch version,
# then install basics with --no-deps so it doesn't try to pull torch 2.6.
!pip install -q vllm==0.7.2 transformers datasets pandas pyarrow \
    latex2sympy2-extended "math-verify[antlr4-13-2]" pylatexenc==2.10 sympy \
    humanfriendly wandb einops einx jaxtyping tiktoken regex psutil submitit
!pip install -q -e ./basics --no-deps

import os
os.kill(os.getpid(), 9)  # force restart so the new torch is picked up

## 2. After restart — sanity check

Run this first after the kernel restarts.

In [ ]:
%cd /content/148bhw2
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
import basics, vllm, transformers
print("basics, vllm, transformers all import OK")

## 3. Systems benchmarks (Section 2.3-2.6)

Sweeps across model sizes + modes. Each run goes to its own `logs/systems/<ts>_*/` folder.

In [ ]:
# 2.3: forward / forward-backward / train-step across sizes
for size in ["small", "medium", "large"]:
    for mode in ["forward", "forward-backward", "train-step"]:
        !python -m systems.benchmark --model-size {size} --mode {mode}

In [ ]:
# 2.5: mixed precision (bf16)
for size in ["small", "medium", "large"]:
    !python -m systems.benchmark --model-size {size} --mode forward-backward --use-bf16

In [ ]:
# 2.6: memory profiler (writes memory.pickle into the run dir; view at https://pytorch.org/memory_viz)
!python -m systems.benchmark --model-size medium --mode train-step --use-memory-profiler
!python -m systems.benchmark --model-size medium --mode train-step --use-memory-profiler --use-bf16

## 4. Attention benchmark (Section 2.7-2.8)

In [ ]:
!python -m systems.attention_benchmark              # eager
!python -m systems.attention_benchmark --compile-attention   # torch.compile

## 5. Alignment baselines (Section 3.1-3.2)

First run downloads Qwen2.5-Math-1.5B (~3GB).

In [ ]:
# Use --limit during dev; remove for a full eval (1319 test examples).
!python -m alignment.eval --mode direct --limit 256
!python -m alignment.eval --mode cot --limit 256
!python -m alignment.eval --mode self_consistency --k 5 --limit 256

## 6. GRPO training (Section 3.5)

Default is 10 steps for a quick sanity check. Bump `--n-grpo-steps` to 50+ for real runs (~30-60 min on H100).

In [ ]:
!python -m alignment.grpo \
    --n-grpo-steps 10 \
    --rollout-batch-size 32 \
    --group-size 8 \
    --train-batch-size 32 \
    --gradient-accumulation-steps 16 \
    --eval-every 5 \
    --n-eval-examples 64 \
    --run-name grpo_smoke

## 7. Bundle logs and download

Colab VMs are ephemeral — pull the logs back to your laptop before the session dies.

In [ ]:
!ls -R logs/ | head -60
# Exclude saved model weights (final/) — they're huge and not needed for the writeup
!cd logs && zip -r /content/logs.zip . -x '*final/*'
from google.colab import files
files.download('/content/logs.zip')